# Eksperimen Fusion Multimodal, versi 2

Versi ini menggantikan v1. Perbedaannya tiga.

1. Protokol latih memakai warmup 10 persen dan cosine decay, batas 8 epoch, patience 3.
2. Urutan dibalik. Fusion diuji lebih dulu pada satu kombinasi acuan, lalu grid encoder
   memakai metode fusion yang menang.
3. Metrik pairwise true lawan MC dilaporkan di samping macro-F1.

Versi 1 tetap ada di `notebooks/02_eksperimen_fusion_v1.ipynb` sebagai pilot dan bukti
pemilihan protokol.

Cara menjalankan:

```bash
SMOKE=1 .venv-train/bin/python experiments/run_notebook.py
FUSION_ONLY=F1 .venv-train/bin/python experiments/run_notebook.py
.venv-train/bin/python experiments/run_notebook.py
```

## 1. Persiapan

**Tujuan** menyiapkan lingkungan, mengunci seed 727, dan menetapkan protokol latih.

**Input** repo ini, `data/07_pairs.jsonl`, dan `experiments/v1/pemenang.json`.

**Output** konfigurasi global, daftar versi library, dan device.

**Keputusan** warmup 10 persen dan cosine decay, 8 epoch, patience 3. Protokol ini
dipilih setelah pilot v1 menunjukkan kurva validasi yang naik-turun tanpa scheduler.

In [1]:
import json
import os
import sys
import time
from collections import defaultdict
from dataclasses import replace
from pathlib import Path
from statistics import median

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader
from transformers import (
    AutoModel,
    AutoTokenizer,
    CLIPVisionModel,
    get_cosine_schedule_with_warmup,
    logging as hf_logging,
)

hf_logging.set_verbosity_error()

ROOT = Path.cwd()
while not (ROOT / "data" / "07_pairs.jsonl").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from experiments.lib.data import (  # noqa: E402
    LABEL_NAMES,
    PairDataset,
    clean_caption,
    load_pairs,
)
from experiments.lib.metrics import (  # noqa: E402
    append_ledger,
    count_params,
    done_ids,
    read_ledger,
    score,
)
from experiments.lib.run import Run, loader_generator, seed_worker, set_seed  # noqa: E402

SMOKE = os.getenv("SMOKE", "0") == "1"
RUN_BASELINE = os.getenv("RUN_BASELINE", "1") == "1"
RUN_FUSION = os.getenv("RUN_FUSION", "1") == "1"
RUN_GRID = os.getenv("RUN_GRID", "0") == "1"
RUN_TUNING = os.getenv("RUN_TUNING", "0") == "1"
RUN_OOF = os.getenv("RUN_OOF", "0") == "1"
RUN_FINAL = os.getenv("RUN_FINAL", "0") == "1"
FUSION_ONLY = os.getenv("FUSION_ONLY", "").strip()
FORCE = os.getenv("FORCE", "0") == "1"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 727
MAX_LEN = 128
IMG_SIZE = 224
WARMUP = 0.1

EXPERIMENTS = ROOT / "experiments"
FIG = EXPERIMENTS / "figures"
RUNS_DIR = EXPERIMENTS / ("runs_smoke" if SMOKE else "runs")
FIG.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
LEDGER = EXPERIMENTS / ("ledger_smoke.csv" if SMOKE else "ledger.csv")
PEMENANG_FUSION = EXPERIMENTS / ("pemenang_fusion_smoke.json" if SMOKE else "pemenang_fusion.json")
PEMENANG = EXPERIMENTS / ("pemenang_smoke.json" if SMOKE else "pemenang.json")

set_seed(SEED)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("device     :", DEVICE)
print("gpu        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")
print("torch      :", torch.__version__)
print("ledger     :", LEDGER.relative_to(ROOT))
print("smoke      :", SMOKE)
print("fusion only:", FUSION_ONLY or "-")

device     : cuda
gpu        : NVIDIA GeForce RTX 4060 Laptop GPU
torch      : 2.14.1+cu130
ledger     : experiments\ledger.csv
smoke      : False
fusion only: -


## 2. Dataset dan EDA

**Tujuan** mengenali isi dataset sebelum modeling.

**Input** `data/07_pairs.jsonl`.

**Output** tabel distribusi label, panjang caption, kategori, dan strata.

**Keputusan** tidak ada. Bab ini deskriptif.

In [2]:
rows = load_pairs()
assert len(rows) == 13674, (
    f"dataset salah: {len(rows)} baris. Clone repo berisi versi lama. "
    "Ganti data/07_pairs.jsonl dengan versi 13.674 baris, "
    "sha256 f9a2f709c0e405e5c2cc9ec4af1c263e86ea8ed55ac4a8e2ee14705dc6416a62."
)
df = pd.DataFrame(rows)
df["n_kata"] = df["caption"].str.split().str.len()

TAB = df["label"].value_counts().rename("jumlah").to_frame()
TAB["porsi"] = (TAB["jumlah"] / len(df)).round(4)
print("total pasangan:", len(df))
display(TAB)
display(df.groupby(["split", "label"]).size().unstack(fill_value=0))
display(df.groupby("fold")["label"].value_counts().unstack(fill_value=0))

total pasangan: 13674


,jumlah,porsi
label,,
true,5689,0.4160
out-of-context,5689,0.4160
miscaptioned,2296,0.1679


label,miscaptioned,out-of-context,true
split,,,
pool,2152,5411,5411
test,144,278,278


label,miscaptioned,out-of-context,true
fold,,,
-1,144,278,278
0,432,1081,1081
1,432,1081,1081
2,431,1082,1082
3,431,1082,1082
4,426,1085,1085


In [3]:
fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
order = ["true", "out-of-context", "miscaptioned"]
counts = df["label"].value_counts().reindex(order)
ax[0].bar(order, counts.values, color=["#3b6ea5", "#7a8b99", "#b5654a"])
ax[0].set_title("Distribusi label")
ax[0].tick_params(axis="x", rotation=20)

for label in order:
    ax[1].hist(df.loc[df.label == label, "n_kata"], bins=40, alpha=0.55, label=label)
ax[1].set_title("Panjang caption (kata)")
ax[1].legend(fontsize=7)

top = df["category"].value_counts().head(10)[::-1]
ax[2].barh(top.index, top.values, color="#3b6ea5")
ax[2].set_title("10 kategori terbesar")
fig.tight_layout()
fig.savefig(FIG / "02_distribusi.png", dpi=160)
plt.close(fig)

print(df.groupby("label")["n_kata"].describe()[["mean", "50%", "max"]].round(2))

                 mean   50%    max
label                             
miscaptioned    25.05  23.0   97.0
out-of-context  22.81  21.0  105.0
true            22.81  21.0  105.0


In [4]:
def stratum(caption: str) -> str:
    import re

    c = caption or ""
    if re.match(r"\s*ilustrasi", c, re.I) or re.search(
        r"(shutterstock|pexels|unsplash|freepik|istock|getty|pixabay|canva|magnific"
        r"|ai generatif|dibuat dengan ai|generated by)",
        c,
        re.I,
    ):
        return "B"
    if re.search(r"(kompas|antara foto|antara/)", c, re.I):
        return "A"
    return "C"


df["stratum"] = df["caption"].map(stratum)
display(df.groupby(["label", "stratum"]).size().unstack(fill_value=0))

stratum,A,B,C
label,,,
miscaptioned,1084,175,1037
out-of-context,2857,587,2245
true,2857,587,2245


## 3. Preprocessing

**Tujuan** menyamakan bentuk input untuk semua encoder.

**Input** caption mentah dan berkas gambar. Gambar diambil dari arsip Drive lewat
`ensure_images()` kalau belum ada, dan fungsi itu idempoten: berkas yang sudah lengkap
tidak diunduh ulang.

**Output** tokenizer IndoBERT dan mBERT, serta gambar 224 piksel dengan letterbox.

**Keputusan** letterbox, bukan center crop, karena rasio asli 16:9. Tanpa augmentasi.

In [5]:
from experiments.lib.images import ensure_images

print(ensure_images())

{'status': 'lengkap', 'jumlah': 5689}


In [6]:
TOKENIZERS = {
    "indobert": AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1"),
    "mbert": AutoTokenizer.from_pretrained("bert-base-multilingual-cased"),
}
print({k: type(v).__name__ for k, v in TOKENIZERS.items()})

from experiments.lib.data import cache_path

missing = sum(1 for r in rows if not cache_path(r["image"]).exists())
assert missing == 0, (
    f"{missing} gambar belum ada di cache. Kirim data/images/ lebih dulu, "
    "lalu jalankan: .venv-train/bin/python -m experiments.build_cache"
)

sample = df.iloc[0]
ids = TOKENIZERS["indobert"](clean_caption(sample["caption"]), truncation=True, max_length=MAX_LEN)
print("contoh caption :", clean_caption(sample["caption"])[:160])
print("jumlah token   :", len(ids["input_ids"]))

{'indobert': 'BertTokenizer', 'mbert': 'BertTokenizer'}
contoh caption : Tampilan baru Halte TransJakarta Senen Sentral pasca berganti nama menjadi Halte Jaga Jakarta di kawasan Pasar Senen, Jakarta Pusat, Selasa (9/9/2025). (KOMPAS.
jumlah token   : 43


In [7]:
from PIL import Image
from experiments.lib.data import cache_path

fig, ax = plt.subplots(1, 2, figsize=(9, 3.6))
with Image.open(cache_path(sample["image"])) as im:
    ax[0].imshow(im)
    ax[0].set_title(f"letterbox cache {im.size[0]}px")
    ax[0].axis("off")
ax[1].text(
    0.0,
    0.5,
    "caption:\n" + sample["caption"][:220] + "\n\n"
    "semua model menerima 224x224,\n"
    "bilah abu-abu di atas dan bawah",
    va="center",
    wrap=True,
    fontsize=9,
)
ax[1].axis("off")
fig.tight_layout()
fig.savefig(FIG / "03_contoh_input.png", dpi=160)
plt.close(fig)

## 4. Pembagian data dan validasi

**Tujuan** memastikan tidak ada kebocoran dan menetapkan peran tiap bagian data.

**Input** kolom `split` dan `fold`.

**Output** latih dan validasi eksplorasi, serta test yang terkunci.

**Keputusan** fold 0 menjadi validasi eksplorasi. Test 700 tidak dipakai sampai model
final.

In [8]:
pool = df[df.split == "pool"].reset_index(drop=True)
test = df[df.split == "test"].reset_index(drop=True)
train_ex = pool[pool.fold != 0].reset_index(drop=True)
val_ex = pool[pool.fold == 0].reset_index(drop=True)

print("pool", len(pool), "| latih", len(train_ex), "| validasi", len(val_ex), "| test", len(test))

def irisan(a: pd.DataFrame, b: pd.DataFrame, col: str) -> int:
    return len(set(a[col]) & set(b[col]))

assert irisan(train_ex, val_ex, "image") == 0
assert irisan(train_ex, val_ex, "caption") == 0
assert irisan(pool, test, "image") == 0
assert irisan(pool, test, "caption") == 0
print("kebocoran gambar dan caption: 0")
print(val_ex.label.value_counts().to_dict())

pool 12974 | latih 10380 | validasi 2594 | test 700
kebocoran gambar dan caption: 0
{'true': 1081, 'out-of-context': 1081, 'miscaptioned': 432}


## 5. Model, loop latih, dan baseline

**Tujuan** menetapkan lantai dan ablasi modalitas sebelum membandingkan strategi fusion.

**Input** split dari bab 4.

**Output** metrik B0, A1 (IndoBERT, mBERT), dan A2 (CLIP, DINOv2).

**Keputusan** B0 tanpa latih. Matriks embedding kata dibekukan untuk menghemat VRAM,
lapisan transformer tetap dilatih, dan perlakuan ini sama untuk semua run teks.

In [9]:
TEXT_NAMES = {
    "indobert": "indobenchmark/indobert-base-p1",
    "mbert": "bert-base-multilingual-cased",
}
CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)
IN_MEAN = (0.485, 0.456, 0.406)
IN_STD = (0.229, 0.224, 0.225)


class TextEncoder(nn.Module):
    def __init__(self, key: str) -> None:
        super().__init__()
        self.enc = AutoModel.from_pretrained(TEXT_NAMES[key])
        self.dim = self.enc.config.hidden_size
        emb = getattr(self.enc, "embeddings", None)
        if emb is not None and hasattr(emb, "word_embeddings"):
            emb.word_embeddings.weight.requires_grad_(False)

    def forward(self, input_ids, attention_mask):
        return self.enc(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state


class ImageEncoder(nn.Module):
    def __init__(self, key: str) -> None:
        super().__init__()
        self.key = key
        if key == "clip":
            self.enc = CLIPVisionModel.from_pretrained("openai/clip-vit-base-patch16")
            self.dim = self.enc.config.hidden_size
            mean, std = CLIP_MEAN, CLIP_STD
        elif key == "dinov2":
            self.enc = timm.create_model(
                "vit_base_patch14_dinov2.lvd142m",
                pretrained=True,
                num_classes=0,
                img_size=IMG_SIZE,
                dynamic_img_size=True,
            )
            self.dim = self.enc.num_features
            mean, std = IN_MEAN, IN_STD
        else:
            raise ValueError(key)
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, pixel):
        pixel = (pixel - self.mean) / self.std
        if self.key == "clip":
            return self.enc(pixel_values=pixel).last_hidden_state
        return self.enc.forward_features(pixel)


class TextOnly(nn.Module):
    def __init__(self, teks: str, dropout: float) -> None:
        super().__init__()
        self.text = TextEncoder(teks)
        d = 256
        self.head = nn.Sequential(
            nn.Linear(self.text.dim, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, 3)
        )

    def forward(self, batch):
        t = self.text(batch["input_ids"], batch["attention_mask"])
        return self.head(t[:, 0])


class ImageOnly(nn.Module):
    def __init__(self, gambar: str, dropout: float) -> None:
        super().__init__()
        self.image = ImageEncoder(gambar)
        d = 256
        self.head = nn.Sequential(
            nn.Linear(self.image.dim, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, 3)
        )

    def forward(self, batch):
        i = self.image(batch["pixel_values"])
        return self.head(i[:, 0])


class ConcatModel(nn.Module):
    def __init__(self, teks: str, gambar: str, dropout: float) -> None:
        super().__init__()
        self.text = TextEncoder(teks)
        self.image = ImageEncoder(gambar)
        d = 256
        self.pt = nn.Linear(self.text.dim, d)
        self.pi = nn.Linear(self.image.dim, d)
        self.head = nn.Sequential(
            nn.LayerNorm(2 * d),
            nn.Linear(2 * d, d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, 3),
        )

    def forward(self, batch):
        t = self.text(batch["input_ids"], batch["attention_mask"])
        i = self.image(batch["pixel_values"])
        z = torch.cat([self.pt(t[:, 0]), self.pi(i[:, 0])], dim=-1)
        return self.head(z)


class LateFusionModel(nn.Module):
    def __init__(self, teks: str, gambar: str, dropout: float) -> None:
        super().__init__()
        self.text = TextEncoder(teks)
        self.image = ImageEncoder(gambar)
        d = 256
        self.ht = nn.Sequential(
            nn.Linear(self.text.dim, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, 3)
        )
        self.hi = nn.Sequential(
            nn.Linear(self.image.dim, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, 3)
        )

    def forward(self, batch):
        t = self.text(batch["input_ids"], batch["attention_mask"])
        i = self.image(batch["pixel_values"])
        return 0.5 * (self.ht(t[:, 0]) + self.hi(i[:, 0]))


class GatedModel(nn.Module):
    def __init__(self, teks: str, gambar: str, dropout: float) -> None:
        super().__init__()
        self.text = TextEncoder(teks)
        self.image = ImageEncoder(gambar)
        d = 256
        self.pt = nn.Linear(self.text.dim, d)
        self.pi = nn.Linear(self.image.dim, d)
        self.gate = nn.Sequential(nn.Linear(2 * d, d), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.LayerNorm(d), nn.Linear(d, d), nn.GELU(), nn.Dropout(dropout), nn.Linear(d, 3)
        )

    def forward(self, batch):
        t = self.pt(self.text(batch["input_ids"], batch["attention_mask"])[:, 0])
        i = self.pi(self.image(batch["pixel_values"])[:, 0])
        g = self.gate(torch.cat([t, i], dim=-1))
        return self.head(g * t + (1 - g) * i)


class CrossAttentionModel(nn.Module):
    def __init__(
        self,
        teks: str,
        gambar: str,
        dropout: float,
        heads: int = 4,
        layers: int = 1,
        bidirectional: bool = True,
    ) -> None:
        super().__init__()
        self.text = TextEncoder(teks)
        self.image = ImageEncoder(gambar)
        d = 256
        self.pt = nn.Linear(self.text.dim, d)
        self.pi = nn.Linear(self.image.dim, d)
        self.t2i = nn.ModuleList(
            [nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True) for _ in range(layers)]
        )
        self.norm_t = nn.ModuleList([nn.LayerNorm(d) for _ in range(layers)])
        self.i2t = (
            nn.ModuleList(
                [nn.MultiheadAttention(d, heads, dropout=dropout, batch_first=True) for _ in range(layers)]
            )
            if bidirectional
            else None
        )
        self.norm_i = nn.ModuleList([nn.LayerNorm(d) for _ in range(layers)]) if bidirectional else None
        self.head = nn.Sequential(
            nn.LayerNorm(2 * d),
            nn.Linear(2 * d, d),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d, 3),
        )

    def forward(self, batch):
        t = self.pt(self.text(batch["input_ids"], batch["attention_mask"]))
        i = self.pi(self.image(batch["pixel_values"]))
        key_pad = ~batch["attention_mask"].bool()
        for k in range(len(self.t2i)):
            a, _ = self.t2i[k](t, i, i, need_weights=False)
            t = self.norm_t[k](t + a)
            if self.i2t is not None:
                b, _ = self.i2t[k](i, t, t, key_padding_mask=key_pad, need_weights=False)
                i = self.norm_i[k](i + b)
        mask = batch["attention_mask"].unsqueeze(-1).float()
        t_pool = (t * mask).sum(1) / mask.sum(1).clamp(min=1)
        i_pool = i.mean(1)
        return self.head(torch.cat([t_pool, i_pool], dim=-1))


def build_model(run: Run) -> nn.Module:
    if run.teks and not run.gambar:
        return TextOnly(run.teks, run.dropout)
    if run.gambar and not run.teks:
        return ImageOnly(run.gambar, run.dropout)
    if run.fusion == "late":
        return LateFusionModel(run.teks, run.gambar, run.dropout)
    if run.fusion == "concat":
        return ConcatModel(run.teks, run.gambar, run.dropout)
    if run.fusion == "gated":
        return GatedModel(run.teks, run.gambar, run.dropout)
    if run.fusion.startswith("xattn"):
        return CrossAttentionModel(
            run.teks,
            run.gambar,
            run.dropout,
            heads=8 if run.fusion == "xattn_bi_deep" else 4,
            layers=2 if run.fusion == "xattn_bi_deep" else 1,
            bidirectional=run.fusion != "xattn_uni",
        )
    raise ValueError(run)


def make_loader(subset, run: Run, shuffle: bool) -> DataLoader:
    tok = TOKENIZERS[run.teks or "indobert"]
    ds = PairDataset(subset, tok, max_len=MAX_LEN, image_size=IMG_SIZE)
    return DataLoader(
        ds,
        batch_size=run.batch,
        shuffle=shuffle,
        num_workers=4,
        pin_memory=True,
        drop_last=False,
        worker_init_fn=seed_worker,
        generator=loader_generator(run.seed),
    )


def move(batch):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}


@torch.no_grad()
def predict(model, loader):
    model.eval()
    probs, ys, ids = [], [], []
    for batch in loader:
        batch = move(batch)
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
            logits = model(batch)
        probs.append(torch.softmax(logits.float(), dim=-1).cpu().numpy())
        ys.append(batch["label"].cpu().numpy())
        ids.append(batch["index"].cpu().numpy())
    return np.concatenate(ys), np.concatenate(probs), np.concatenate(ids)


def class_weights(subset) -> torch.Tensor:
    counts = pd.Series([r["label"] for r in subset]).value_counts()
    w = torch.tensor([len(subset) / (3 * counts.get(name, 1)) for name in LABEL_NAMES], dtype=torch.float)
    return w.to(DEVICE)


def build_pairs(subset) -> list[tuple[int, int]]:
    idx = defaultdict(list)
    for i, r in enumerate(subset):
        idx[(r["source"], r["image"])].append(i)
    pairs = []
    for _, ids in idx.items():
        ts = [i for i in ids if subset[i]["label"] == "true"]
        ms = [i for i in ids if subset[i]["label"] == "miscaptioned"]
        if ts and ms:
            for m in ms:
                pairs.append((ts[0], m))
    return pairs


def pairwise_acc(run_id: str, pairs: list[tuple[int, int]]) -> float:
    path = RUNS_DIR / f"{run_id}-val.npz"
    if not pairs or not path.exists():
        return float("nan")
    p = np.load(path)["prob"][:, 0]
    return float(np.mean([p[a] > p[b] for a, b in pairs]))


def run_one(run: Run, train_rows, val_rows, epochs: int | None = None, save: bool = True) -> dict:
    if not FORCE and run.run_id in done_ids(LEDGER):
        print("lewati, sudah ada:", run.run_id)
        return next(r for r in read_ledger(LEDGER) if r["run_id"] == run.run_id)

    set_seed(run.seed)
    train_loader = make_loader(train_rows, run, shuffle=True)
    val_loader = make_loader(val_rows, run, shuffle=False)
    model = build_model(run).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=class_weights(train_rows))
    opt = torch.optim.AdamW(model.parameters(), lr=run.lr, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")

    max_epochs = epochs or run.max_epochs
    steps = len(train_loader) * max_epochs
    sched = get_cosine_schedule_with_warmup(
        opt, num_warmup_steps=int(WARMUP * steps), num_training_steps=steps
    )

    best_f1, best_epoch, best_state, bad = -1.0, 0, None, 0
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        for batch in train_loader:
            batch = move(batch)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                loss = crit(model(batch), batch["label"])
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()
            sched.step()
        y, p, _ = predict(model, val_loader)
        f1 = f1_score(y, p.argmax(1), average="macro", labels=[0, 1, 2], zero_division=0)
        print(f"  {run.run_id} epoch {epoch} f1_macro {f1:.4f}")
        if f1 > best_f1:
            best_f1, best_epoch, bad = f1, epoch, 0
            best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= run.patience:
                break

    wall = time.perf_counter() - t0
    vram = torch.cuda.max_memory_allocated() / 1e6 if DEVICE.type == "cuda" else 0.0
    if best_state is not None:
        model.load_state_dict(best_state)
    y, p, ids = predict(model, val_loader)
    sc = score(y, p.argmax(1))

    if save:
        torch.save(best_state, RUNS_DIR / f"{run.run_id}.pt")
        np.savez(RUNS_DIR / f"{run.run_id}-val.npz", ids=ids, y=y, prob=p)

    row = run.row() | {
        "epochs_run": epoch,
        "epoch_best": best_epoch,
        "acc": round(sc["acc"], 4),
        "bal_acc": round(sc["bal_acc"], 4),
        "f1_macro": round(sc["f1_macro"], 4),
        "f1_true": round(sc["f1_true"], 4),
        "f1_ooc": round(sc["f1_ooc"], 4),
        "f1_mc": round(sc["f1_mc"], 4),
        "val_n": len(val_rows),
        "wall_clock_s": round(wall, 1),
        "params": count_params(model),
        "vram_peak_mb": round(vram, 0),
        "catatan": "fold0-val",
    }
    append_ledger(row, LEDGER)
    del model, best_state, opt, crit, scaler, train_loader, val_loader, sched
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return row

In [10]:
BASE = dict(lr=2e-5, dropout=0.1, batch=16, seed=SEED, max_epochs=8, patience=3)
RUNS_BASELINE = [
    Run("A1", "modalitas", "indobert", None, "none", **BASE),
    Run("A1", "modalitas", "mbert", None, "none", **BASE),
    Run("A2", "modalitas", None, "clip", "none", **BASE),
    Run("A2", "modalitas", None, "dinov2", "none", **BASE),
]

if SMOKE:
    train_rows = train_ex.groupby("label").head(100).to_dict("records")
    val_eval = val_ex.groupby("label").head(50)
    val_rows = val_eval.to_dict("records")
    RUNS_BASELINE = [replace(r, max_epochs=1, patience=1) for r in RUNS_BASELINE]
else:
    train_rows = train_ex.to_dict("records")
    val_rows = val_ex.to_dict("records")
    val_eval = val_ex

PAIRS = build_pairs(val_rows)

mayor = pd.Series([r["label"] for r in train_rows]).value_counts().idxmax()
b0_pred = np.full(len(val_rows), LABEL_NAMES.index(mayor))
b0 = score(val_eval.label.map(LABEL_NAMES.index).to_numpy(), b0_pred)
b0_run = Run("B0", "modalitas", None, None, "none", **BASE)
if FORCE or b0_run.run_id not in done_ids(LEDGER):
    append_ledger(
        b0_run.row()
        | {
            "acc": round(b0["acc"], 4),
            "bal_acc": round(b0["bal_acc"], 4),
            "f1_macro": round(b0["f1_macro"], 4),
            "f1_true": round(b0["f1_true"], 4),
            "f1_ooc": round(b0["f1_ooc"], 4),
            "f1_mc": round(b0["f1_mc"], 4),
            "val_n": len(val_rows),
            "catatan": f"mayoritas={mayor}",
        },
        LEDGER,
    )
print("B0", {k: round(b0[k], 4) for k in ["acc", "f1_macro", "f1_mc"]})

if RUN_BASELINE:
    for run in RUNS_BASELINE:
        run_one(run, train_rows, val_rows)
else:
    print("RUN_BASELINE mati")

B0 {'acc': 0.4167, 'f1_macro': 0.1961, 'f1_mc': 0.0}
lewati, sudah ada: A1_indobert_na_none_s727
lewati, sudah ada: A1_mbert_na_none_s727
lewati, sudah ada: A2_na_clip_none_s727
lewati, sudah ada: A2_na_dinov2_none_s727


## 6. Perbandingan strategi fusion

**Tujuan** menguji apakah interaksi antar modalitas menambah performa, dan pada level
mana.

**Input** kombinasi acuan dari pilot v1, yaitu M4 mBERT dan DINOv2.

**Output** enam run F0 sampai F5.

**Keputusan** kombinasi encoder dikunci, yang berubah hanya fusion. Urutannya naik dari
keputusan, ke fitur global, ke fitur berbobot, ke token.

In [11]:
REF_PATH = EXPERIMENTS / "v1" / "pemenang.json"
if REF_PATH.exists():
    REF = json.loads(REF_PATH.read_text())
else:
    REF = {"kode": "M4", "teks": "mbert", "gambar": "dinov2"}
print("kombinasi acuan:", REF["kode"], REF["teks"], REF["gambar"])

FUSIONS = [
    ("F0", "late"),
    ("F1", "concat"),
    ("F2", "gated"),
    ("F3", "xattn_uni"),
    ("F4", "xattn_bi"),
    ("F5", "xattn_bi_deep"),
]
RUNS_FUSION = [
    Run(kode, "fusion", REF["teks"], REF["gambar"], fusion, **BASE) for kode, fusion in FUSIONS
]
if FUSION_ONLY:
    RUNS_FUSION = [r for r in RUNS_FUSION if r.kode == FUSION_ONLY]
if SMOKE:
    RUNS_FUSION = [replace(r, max_epochs=1, patience=1) for r in RUNS_FUSION]

if RUN_FUSION:
    for run in RUNS_FUSION:
        run_one(run, train_rows, val_rows)
else:
    print("RUN_FUSION mati")

kombinasi acuan: M4 mbert dinov2
lewati, sudah ada: F0_mbert_dinov2_late_s727
lewati, sudah ada: F1_mbert_dinov2_concat_s727
lewati, sudah ada: F2_mbert_dinov2_gated_s727
lewati, sudah ada: F3_mbert_dinov2_xattn_uni_s727
lewati, sudah ada: F4_mbert_dinov2_xattn_bi_s727
lewati, sudah ada: F5_mbert_dinov2_xattn_bi_deep_s727


## 7. Analisis fusion dan keputusan

**Tujuan** memilih metode fusion yang dipakai di grid encoder.

**Input** ledger berisi F0 sampai F5.

**Output** tabel, grafik, dan `experiments/pemenang_fusion.json`.

**Keputusan** pemenang diambil dari `f1_macro` tertinggi. Kalau selisih dua teratas di
bawah 0,02, keduanya dianggap seri dan pemenang ditentukan oleh kesesuaian dengan
pertanyaan penelitian, yaitu cross-attention satu arah, selama skornya masih di dalam
0,02 dari yang tertinggi. Aturan ini dinyatakan sebelum grid dijalankan, supaya
pemilihan tidak bergantung pada hasil akhir.

In [12]:
led = pd.DataFrame(read_ledger(LEDGER))
for col in ["acc", "bal_acc", "f1_macro", "f1_true", "f1_ooc", "f1_mc"]:
    led[col] = pd.to_numeric(led[col], errors="coerce")

fus = led[led.kode.isin([f"F{i}" for i in range(6)])].drop_duplicates("kode").copy()
fus["pairwise"] = [pairwise_acc(r, PAIRS) for r in fus["run_id"]]
display(fus[["kode", "fusion", "acc", "bal_acc", "f1_macro", "f1_mc", "pairwise", "epoch_best"]])

if len(fus):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    d = fus.sort_values("f1_macro")
    ax[0].barh(d["kode"] + " " + d["fusion"], d["f1_macro"], color="#3b6ea5")
    ax[0].set_title("macro-F1 per strategi fusion")
    ax[1].barh(d["kode"] + " " + d["fusion"], d["pairwise"], color="#b5654a")
    ax[1].set_xlim(0.5, 1.0)
    ax[1].set_title("pairwise true lawan MC")
    fig.tight_layout()
    fig.savefig(FIG / "07_fusion.png", dpi=160)
    plt.close(fig)

    ranked = fus.sort_values("f1_macro", ascending=False).reset_index(drop=True)
    gap = float(ranked.loc[0, "f1_macro"] - ranked.loc[1, "f1_macro"]) if len(ranked) > 1 else 0.0
    best = ranked.loc[0]
    aturan = "macro-F1 tertinggi"

    # Tie-break yang dinyatakan sebelum grid: kalau selisih dua teratas di bawah 0,02,
    # pilih strategi yang menjadi fokus penelitian selama skornya masih di dalam 0,02.
    STUDI = "xattn_uni"
    if gap < 0.02:
        kandidat = fus[fus.fusion == STUDI]
        if len(kandidat):
            k = kandidat.iloc[0]
            if float(best["f1_macro"]) - float(k["f1_macro"]) < 0.02:
                best = k
                aturan = "seri, dipilih strategi penelitian"

    pemenang_fusion = {
        "kode": str(best["kode"]),
        "fusion": str(best["fusion"]),
        "f1_macro": float(best["f1_macro"]),
        "pairwise": float(best["pairwise"]) if pd.notna(best["pairwise"]) else None,
        "epoch_best": int(best["epoch_best"]),
        "selisih_dua_teratas": round(gap, 4),
        "aturan": aturan,
        "acuan_encoder": {"teks": REF["teks"], "gambar": REF["gambar"]},
        "ledger": LEDGER.name,
    }
    PEMENANG_FUSION.write_text(json.dumps(pemenang_fusion, indent=2))
    print(json.dumps(pemenang_fusion, indent=2))
else:
    print("belum ada hasil fusion")

,kode,fusion,acc,bal_acc,f1_macro,f1_mc,pairwise,epoch_best
1,F1,concat,0.5289,0.5616,0.5466,0.6275,0.752315,8
6,F0,late,0.4861,0.5357,0.4999,0.6167,0.736111,6
7,F2,gated,0.5671,0.5865,0.5754,0.6152,0.708333,8
8,F3,xattn_uni,0.5563,0.5853,0.5696,0.6339,0.715278,8
9,F4,xattn_bi,0.4927,0.5465,0.5019,0.5949,0.731481,3
10,F5,xattn_bi_deep,0.5274,0.5590,0.5422,0.6135,0.733796,8


{
  "kode": "F3",
  "fusion": "xattn_uni",
  "f1_macro": 0.5696,
  "pairwise": 0.7152777777777778,
  "epoch_best": 8,
  "selisih_dua_teratas": 0.0058,
  "aturan": "seri, dipilih strategi penelitian",
  "acuan_encoder": {
    "teks": "mbert",
    "gambar": "dinov2"
  },
  "ledger": "ledger.csv"
}


## 8. Grid encoder dengan fusion pemenang

**Tujuan** membandingkan empat kombinasi encoder memakai metode fusion yang menang.

**Input** `experiments/pemenang_fusion.json`.

**Output** empat run M1 sampai M4, tabel, grafik, dan `experiments/pemenang.json`.

**Keputusan** fusion dikunci ke pemenang bab 7. Hyperparameter dan seed sama untuk
keempat. Yang berubah hanya pasangan encoder.

**Kontrak** helper dari bab 5 sampai 7 sudah tersedia, yaitu `Run`, `run_one`,
`build_model`, `make_loader`, `pairwise_acc`, `train_rows`, `val_rows`, `pool`,
`test`, dan `BASE`. Artefak yang harus dihasilkan di bab ini sampai bab 12:
`experiments/pemenang.json`, baris baru di ledger, `runs/oof.npz`, dan
`runs/Z*-test.npz`. Protokol tidak boleh diubah. Contoh pemanggilan:
`run_one(Run("M1", "encoder", "indobert", "clip", FUS, **BASE), train_rows, val_rows)`.

In [12]:
fus_path = PEMENANG_FUSION if PEMENANG_FUSION.exists() else EXPERIMENTS / "pemenang_fusion.json"
fus_cfg = json.loads(fus_path.read_text())
FUS = fus_cfg["fusion"]
print("fusion pemenang:", fus_cfg["kode"], FUS)

RUNS_GRID = [
    Run("M1", "encoder", "indobert", "clip", FUS, **BASE),
    Run("M2", "encoder", "indobert", "dinov2", FUS, **BASE),
    Run("M3", "encoder", "mbert", "clip", FUS, **BASE),
    Run("M4", "encoder", "mbert", "dinov2", FUS, **BASE),
]
if SMOKE:
    RUNS_GRID = [replace(r, max_epochs=1, patience=1) for r in RUNS_GRID]

if os.getenv("RUN_GRID", "1") == "1":
    for run in RUNS_GRID:
        run_one(run, train_rows, val_rows)
else:
    print("RUN_GRID mati")

led = pd.DataFrame(read_ledger(LEDGER))
for col in ["acc", "bal_acc", "f1_macro", "f1_true", "f1_ooc", "f1_mc", "epoch_best", "lr", "dropout", "batch", "seed", "val_n"]:
    led[col] = pd.to_numeric(led[col], errors="coerce")

tahap1 = led[led.kode.isin(["B0", "A1", "A2", "M1", "M2", "M3", "M4"])].drop_duplicates("run_id").copy()
grid = led[led.kode.isin(["M1", "M2", "M3", "M4"])].drop_duplicates("kode").copy()
if len(grid):
    grid["pairwise"] = [pairwise_acc(r, PAIRS) for r in grid["run_id"]]
    grid_sorted = grid.sort_values("f1_macro", ascending=False).reset_index(drop=True)
    display(grid_sorted[["kode", "teks", "gambar", "fusion", "acc", "bal_acc", "f1_macro", "f1_mc", "pairwise", "epoch_best"]])

    fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8))
    d_all = tahap1.sort_values("f1_macro")
    labels_all = [f"{r.kode} ({r.teks or '-'} / {r.gambar or '-'})" for r in d_all.itertuples()]
    ax[0].barh(labels_all, d_all["f1_macro"], color="#3b6ea5")
    ax[0].axvline(0.1961, color="#b5654a", linestyle="--", linewidth=1, label="B0 (0.196)")
    ax[0].set_xlabel("macro-F1")
    ax[0].set_title(f"Ablasi & Grid Encoder ({FUS})")
    ax[0].legend(fontsize=8)

    d_grid = grid.sort_values("pairwise")
    ax[1].barh(d_grid["kode"] + " (" + d_grid["teks"] + "+" + d_grid["gambar"] + ")", d_grid["pairwise"], color="#b5654a")
    ax[1].set_xlim(0.5, 1.0)
    ax[1].axvline(0.810, color="#3b6ea5", linestyle="--", linewidth=1, label="pilot teks (0.810)")
    ax[1].set_xlabel("pairwise true vs MC")
    ax[1].set_title("Pairwise Grid Encoder")
    ax[1].legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG / ("08_grid_smoke.png" if SMOKE else "08_grid.png"), dpi=160)
    plt.close(fig)

    gap = float(grid_sorted.loc[0, "f1_macro"] - grid_sorted.loc[1, "f1_macro"]) if len(grid_sorted) > 1 else 0.0
    best = grid_sorted.loc[0]
    if gap < 0.02 and len(grid_sorted) > 1:
        print(
            f"catatan: selisih dua teratas ({grid_sorted.loc[0, 'kode']} vs {grid_sorted.loc[1, 'kode']}) "
            f"= {gap:.4f} < 0,02"
        )

    pemenang = {
        "kode": str(best["kode"]),
        "teks": str(best["teks"]),
        "gambar": str(best["gambar"]),
        "fusion": str(best["fusion"]),
        "lr": float(best["lr"]),
        "dropout": float(best["dropout"]),
        "batch": int(best["batch"]),
        "seed": int(best["seed"]),
        "epoch_best": int(best["epoch_best"]),
        "f1_macro": float(best["f1_macro"]),
        "pairwise": float(best["pairwise"]) if pd.notna(best["pairwise"]) else None,
        "selisih_dua_teratas": round(gap, 4),
        "ledger": LEDGER.name,
        "val_n": int(best["val_n"]),
    }
    PEMENANG.write_text(json.dumps(pemenang, indent=2))
    print(json.dumps(pemenang, indent=2))
else:
    print("belum ada hasil grid M1..M4")

fusion pemenang: F3 xattn_uni


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  M1_indobert_clip_xattn_uni_s727 epoch 1 f1_macro 0.4682
  M1_indobert_clip_xattn_uni_s727 epoch 2 f1_macro 0.5139
  M1_indobert_clip_xattn_uni_s727 epoch 3 f1_macro 0.4962
  M1_indobert_clip_xattn_uni_s727 epoch 4 f1_macro 0.4434
  M1_indobert_clip_xattn_uni_s727 epoch 5 f1_macro 0.5164
  M1_indobert_clip_xattn_uni_s727 epoch 6 f1_macro 0.6107
  M1_indobert_clip_xattn_uni_s727 epoch 7 f1_macro 0.6343
  M1_indobert_clip_xattn_uni_s727 epoch 8 f1_macro 0.6346


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\ACER\AppData\Local\Temp\ipykernel_13728\3703766489.py:315: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


... (5 outputs left out)

C:\Users\ACER\AppData\Local\Temp\ipykernel_13728\3703766489.py:315: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  sched.step()


  M4_mbert_dinov2_xattn_uni_s727 epoch 1 f1_macro 0.4423
  M4_mbert_dinov2_xattn_uni_s727 epoch 2 f1_macro 0.4932
  M4_mbert_dinov2_xattn_uni_s727 epoch 3 f1_macro 0.5001
  M4_mbert_dinov2_xattn_uni_s727 epoch 4 f1_macro 0.4064
  M4_mbert_dinov2_xattn_uni_s727 epoch 5 f1_macro 0.5061
  M4_mbert_dinov2_xattn_uni_s727 epoch 6 f1_macro 0.5012
  M4_mbert_dinov2_xattn_uni_s727 epoch 7 f1_macro 0.5640
  M4_mbert_dinov2_xattn_uni_s727 epoch 8 f1_macro 0.5690


,kode,teks,gambar,fusion,acc,bal_acc,f1_macro,f1_mc,pairwise,epoch_best
0,M1,indobert,clip,xattn_uni,0.6268,0.6408,0.6346,0.6810,0.763889,8.0
1,M2,indobert,dinov2,xattn_uni,0.5991,0.6163,0.6136,0.6789,0.784722,8.0
2,M4,mbert,dinov2,xattn_uni,0.5528,0.5844,0.5690,0.6443,0.747685,8.0
3,M3,mbert,clip,xattn_uni,0.5459,0.5770,0.5606,0.6280,0.740741,8.0


{
  "kode": "M1",
  "teks": "indobert",
  "gambar": "clip",
  "fusion": "xattn_uni",
  "lr": 2e-05,
  "dropout": 0.1,
  "batch": 16,
  "seed": 727,
  "epoch_best": 8,
  "f1_macro": 0.6346,
  "pairwise": 0.7638888888888888,
  "selisih_dua_teratas": 0.021,
  "ledger": "ledger.csv",
  "val_n": 2594
}


## 9. Faktor hyperparameter

**Tujuan** mencari learning rate dan dropout terbaik pada konfigurasi final.

**Input** pemenang encoder dan fusion.

**Output** enam run T1 sampai T6.

**Keputusan** grid dua parameter, bukan OFAT.

In [13]:
P = json.loads(PEMENANG.read_text())
print("pemenang encoder:", P["kode"], P["teks"], P["gambar"], P["fusion"])

RUNS_TUNING = []
idx = 1
for lr in [1e-5, 2e-5, 5e-5]:
    for do in [0.1, 0.3]:
        RUNS_TUNING.append(
            Run(
                f"T{idx}",
                "tuning",
                P["teks"],
                P["gambar"],
                P["fusion"],
                lr=lr,
                dropout=do,
                batch=int(P["batch"]),
                seed=SEED,
            )
        )
        idx += 1

if SMOKE:
    RUNS_TUNING = [replace(r, max_epochs=1, patience=1) for r in RUNS_TUNING[:2]]

if os.getenv("RUN_TUNING", "1") == "1":
    for run in RUNS_TUNING:
        run_one(run, train_rows, val_rows)
else:
    print("RUN_TUNING mati")

led = pd.DataFrame(read_ledger(LEDGER))
for col in ["lr", "dropout", "acc", "bal_acc", "f1_macro", "f1_true", "f1_ooc", "f1_mc", "epoch_best"]:
    led[col] = pd.to_numeric(led[col], errors="coerce")

tun = led[led.kode.isin([f"T{i}" for i in range(1, 7)])].drop_duplicates("kode").copy()
if len(tun):
    tun["pairwise"] = [pairwise_acc(r, PAIRS) for r in tun["run_id"]]
    tun = tun.sort_values("f1_macro", ascending=False).reset_index(drop=True)
    display(tun[["kode", "lr", "dropout", "acc", "bal_acc", "f1_macro", "f1_mc", "pairwise", "epoch_best"]])

    fig, ax = plt.subplots(figsize=(7.5, 3.6))
    d = tun.sort_values("f1_macro")
    ax.barh(d["kode"] + " lr=" + d["lr"].astype(str) + " do=" + d["dropout"].astype(str), d["f1_macro"], color="#3b6ea5")
    ax.set_xlabel("macro-F1")
    ax.set_title(f"Tuning {P['teks']}+{P['gambar']} ({P['fusion']})")
    fig.tight_layout()
    fig.savefig(FIG / ("09_tuning_smoke.png" if SMOKE else "09_tuning.png"), dpi=160)
    plt.close(fig)

    best_t = tun.iloc[0]
    FINAL_LR = float(best_t["lr"])
    FINAL_DO = float(best_t["dropout"])
    print("tuning terbaik:", best_t["kode"], "| lr =", FINAL_LR, "| dropout =", FINAL_DO, "| f1_macro =", best_t["f1_macro"])
else:
    FINAL_LR = float(P["lr"])
    FINAL_DO = float(P["dropout"])
    print("belum ada hasil tuning, pakai default pemenang:", FINAL_LR, FINAL_DO)

pemenang encoder: M1 indobert clip xattn_uni


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  T1_indobert_clip_xattn_uni_s727 epoch 1 f1_macro 0.4868
  T1_indobert_clip_xattn_uni_s727 epoch 2 f1_macro 0.6204
  T1_indobert_clip_xattn_uni_s727 epoch 3 f1_macro 0.6260
  T1_indobert_clip_xattn_uni_s727 epoch 4 f1_macro 0.6516
  T1_indobert_clip_xattn_uni_s727 epoch 5 f1_macro 0.6756
  T1_indobert_clip_xattn_uni_s727 epoch 6 f1_macro 0.6882
  T1_indobert_clip_xattn_uni_s727 epoch 7 f1_macro 0.6930
  T1_indobert_clip_xattn_uni_s727 epoch 8 f1_macro 0.6951


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (11 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  T6_indobert_clip_xattn_uni_s727 epoch 1 f1_macro 0.3608
  T6_indobert_clip_xattn_uni_s727 epoch 2 f1_macro 0.2396
  T6_indobert_clip_xattn_uni_s727 epoch 3 f1_macro 0.3966
  T6_indobert_clip_xattn_uni_s727 epoch 4 f1_macro 0.3878
  T6_indobert_clip_xattn_uni_s727 epoch 5 f1_macro 0.4003
  T6_indobert_clip_xattn_uni_s727 epoch 6 f1_macro 0.4252
  T6_indobert_clip_xattn_uni_s727 epoch 7 f1_macro 0.5127
  T6_indobert_clip_xattn_uni_s727 epoch 8 f1_macro 0.4862


,kode,lr,dropout,acc,bal_acc,f1_macro,f1_mc,pairwise,epoch_best
0,T1,0.00001,0.1,0.6916,0.6922,0.6951,0.7126,0.777778,8.0
1,T2,0.00001,0.3,0.6904,0.6885,0.6915,0.6983,0.798611,8.0
2,T4,0.00002,0.3,0.6056,0.6243,0.6221,0.6895,0.782407,8.0
3,T3,0.00002,0.1,0.5281,0.5623,0.5581,0.6887,0.761574,7.0
4,T6,0.00005,0.3,0.4911,0.5253,0.5127,0.6204,0.782407,7.0
5,T5,0.00005,0.1,0.4776,0.4729,0.4589,0.5255,0.611111,3.0


tuning terbaik: T1 | lr = 1e-05 | dropout = 0.1 | f1_macro = 0.6951


## 10. Final 5-fold OOF

**Tujuan** mengukur stabilitas dan menyiapkan prediksi out-of-fold untuk pairwise.

**Input** konfigurasi final.

**Output** lima run O1 sampai O5, `oof.npz`, dan tabel per fold.

**Keputusan** satu seed, early stopping tetap dipakai, median `epoch_best` jadi epoch
model final.

In [12]:
P = json.loads(PEMENANG.read_text())
led_t = pd.DataFrame(read_ledger(LEDGER))
led_t["f1_macro"] = pd.to_numeric(led_t["f1_macro"], errors="coerce")
led_t["lr"] = pd.to_numeric(led_t["lr"], errors="coerce")
led_t["dropout"] = pd.to_numeric(led_t["dropout"], errors="coerce")
tun_rows = led_t[led_t.kode.isin([f"T{i}" for i in range(1, 7)])].sort_values("f1_macro", ascending=False)
if len(tun_rows):
    FINAL_LR = float(tun_rows.iloc[0]["lr"])
    FINAL_DO = float(tun_rows.iloc[0]["dropout"])
else:
    FINAL_LR = float(P["lr"])
    FINAL_DO = float(P["dropout"])
print("konfigurasi OOF:", P["teks"], P["gambar"], P["fusion"], "| lr =", FINAL_LR, "| dropout =", FINAL_DO)

OOF_IDS = [
    Run(
        f"O{k + 1}",
        "oof",
        P["teks"],
        P["gambar"],
        P["fusion"],
        lr=FINAL_LR,
        dropout=FINAL_DO,
        batch=int(P["batch"]),
        seed=SEED,
        max_epochs=BASE["max_epochs"],
        patience=BASE["patience"],
    )
    for k in range(5)
]
if SMOKE:
    OOF_IDS = [replace(r, max_epochs=1, patience=1) for r in OOF_IDS]

oof_fold_dfs = []
for k in range(5):
    va_df = pool[pool.fold == k].reset_index(drop=True)
    if SMOKE:
        va_df = va_df.groupby("label").head(50).reset_index(drop=True)
    oof_fold_dfs.append(va_df)

if os.getenv("RUN_OOF", "1") == "1":
    for k, run in enumerate(OOF_IDS):
        tr_df = pool[pool.fold != k].reset_index(drop=True)
        if SMOKE:
            tr_df = tr_df.groupby("label").head(100).reset_index(drop=True)
        run_one(run, tr_df.to_dict("records"), oof_fold_dfs[k].to_dict("records"))

    parts = [np.load(RUNS_DIR / f"{r.run_id}-val.npz") for r in OOF_IDS if (RUNS_DIR / f"{r.run_id}-val.npz").exists()]
    if len(parts) == 5:
        np.savez(
            RUNS_DIR / "oof.npz",
            ids=np.concatenate([p["ids"] for p in parts]),
            y=np.concatenate([p["y"] for p in parts]),
            prob=np.concatenate([p["prob"] for p in parts]),
            fold=np.concatenate([np.full(len(p["y"]), k) for k, p in enumerate(parts)]),
        )
        print("oof.npz tersimpan, total baris:", sum(len(p["y"]) for p in parts))
else:
    print("RUN_OOF mati")

led = pd.DataFrame(read_ledger(LEDGER))
for col in ["acc", "bal_acc", "f1_macro", "f1_true", "f1_ooc", "f1_mc", "epoch_best"]:
    led[col] = pd.to_numeric(led[col], errors="coerce")

oof_led = led[led.kode.isin([f"O{k + 1}" for k in range(5)])].drop_duplicates("kode").sort_values("kode").copy()
if len(oof_led):
    pw_map = {
        r.kode: pairwise_acc(r.run_id, build_pairs(oof_fold_dfs[k].to_dict("records")))
        for k, r in enumerate(OOF_IDS)
    }
    oof_led["pairwise"] = oof_led["kode"].map(pw_map)
    display(oof_led[["kode", "acc", "bal_acc", "f1_macro", "f1_mc", "pairwise", "epoch_best"]])

    summary_oof = pd.DataFrame({
        "mean": oof_led[["acc", "bal_acc", "f1_macro", "f1_mc", "pairwise"]].mean().round(4),
        "std": oof_led[["acc", "bal_acc", "f1_macro", "f1_mc", "pairwise"]].std(ddof=1).round(4),
    })
    display(summary_oof)

    EPOCHS_FINAL = max(1, int(median(oof_led["epoch_best"].dropna().tolist())))
    print("median epoch_best OOF -> EPOCHS_FINAL =", EPOCHS_FINAL)

    fig, ax = plt.subplots(figsize=(7.2, 3.4))
    ax.bar(oof_led["kode"], oof_led["f1_macro"], color="#3b6ea5")
    ax.axhline(oof_led["f1_macro"].mean(), color="#b5654a", linestyle="--", linewidth=1, label=f"mean={oof_led['f1_macro'].mean():.4f}")
    ax.set_ylabel("macro-F1")
    ax.set_title("5-Fold OOF per Fold")
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG / ("10_oof_smoke.png" if SMOKE else "10_oof.png"), dpi=160)
    plt.close(fig)
else:
    EPOCHS_FINAL = int(P["epoch_best"])
    print("belum ada hasil OOF, fallback EPOCHS_FINAL =", EPOCHS_FINAL)

konfigurasi OOF: indobert clip xattn_uni | lr = 1e-05 | dropout = 0.1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  O1_indobert_clip_xattn_uni_s727 epoch 1 f1_macro 0.5023
  O1_indobert_clip_xattn_uni_s727 epoch 2 f1_macro 0.6429
  O1_indobert_clip_xattn_uni_s727 epoch 3 f1_macro 0.6408
  O1_indobert_clip_xattn_uni_s727 epoch 4 f1_macro 0.6654
  O1_indobert_clip_xattn_uni_s727 epoch 5 f1_macro 0.6902
  O1_indobert_clip_xattn_uni_s727 epoch 6 f1_macro 0.6990
  O1_indobert_clip_xattn_uni_s727 epoch 7 f1_macro 0.6987
  O1_indobert_clip_xattn_uni_s727 epoch 8 f1_macro 0.6988


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (9 outputs left out)

  O5_indobert_clip_xattn_uni_s727 epoch 1 f1_macro 0.5175
  O5_indobert_clip_xattn_uni_s727 epoch 2 f1_macro 0.5963
  O5_indobert_clip_xattn_uni_s727 epoch 3 f1_macro 0.6635
  O5_indobert_clip_xattn_uni_s727 epoch 4 f1_macro 0.6633
  O5_indobert_clip_xattn_uni_s727 epoch 5 f1_macro 0.6831
  O5_indobert_clip_xattn_uni_s727 epoch 6 f1_macro 0.6963
  O5_indobert_clip_xattn_uni_s727 epoch 7 f1_macro 0.7012
  O5_indobert_clip_xattn_uni_s727 epoch 8 f1_macro 0.7009
oof.npz tersimpan, total baris: 12974


,kode,acc,bal_acc,f1_macro,f1_mc,pairwise,epoch_best
21,O1,0.6951,0.6982,0.6990,0.7156,0.796296,6.0
22,O2,0.7136,0.7273,0.7227,0.7630,0.868056,8.0
23,O3,0.6936,0.7058,0.6981,0.7214,0.835267,7.0
24,O4,0.7171,0.7208,0.7222,0.7459,0.832947,7.0
25,O5,0.6911,0.7051,0.7012,0.7448,0.854460,7.0


,mean,std
acc,0.7021,0.0122
bal_acc,0.7114,0.0121
f1_macro,0.7086,0.0127
f1_mc,0.7381,0.0194
pairwise,0.8374,0.0271


median epoch_best OOF -> EPOCHS_FINAL = 7


## 11. Model final dan uji 700

**Tujuan** menghasilkan angka utama.

**Input** seluruh pool, test 700, dan epoch dari median OOF.

**Output** tiga run Z1 sampai Z3, prediksi test, rata-rata dan simpangan.

**Keputusan** seed 727, 728, 729, tanpa validasi, epoch tetap.

In [13]:
P = json.loads(PEMENANG.read_text())
led = pd.DataFrame(read_ledger(LEDGER))
for col in ["f1_macro", "lr", "dropout", "epoch_best"]:
    led[col] = pd.to_numeric(led[col], errors="coerce")

tun_rows = led[led.kode.isin([f"T{i}" for i in range(1, 7)])].sort_values("f1_macro", ascending=False)
FINAL_LR = float(tun_rows.iloc[0]["lr"]) if len(tun_rows) else float(P["lr"])
FINAL_DO = float(tun_rows.iloc[0]["dropout"]) if len(tun_rows) else float(P["dropout"])

oof_rows = led[led.kode.isin([f"O{k + 1}" for k in range(5)])].drop_duplicates("kode")
EPOCHS_FINAL = max(1, int(median(oof_rows["epoch_best"].dropna().tolist()))) if len(oof_rows) else int(P["epoch_best"])
print("final config:", P["teks"], P["gambar"], P["fusion"], "| lr =", FINAL_LR, "| do =", FINAL_DO, "| epochs =", EPOCHS_FINAL)

epochs_final = 1 if SMOKE else EPOCHS_FINAL
FINAL_SEEDS = [SEED, SEED + 1, SEED + 2]
FINAL = [
    Run(
        f"Z{i + 1}",
        "final",
        P["teks"],
        P["gambar"],
        P["fusion"],
        lr=FINAL_LR,
        dropout=FINAL_DO,
        batch=int(P["batch"]),
        seed=s,
        max_epochs=epochs_final,
        patience=BASE["patience"],
    )
    for i, s in enumerate(FINAL_SEEDS)
]

pool_df = pool.groupby("label").head(100).reset_index(drop=True) if SMOKE else pool.reset_index(drop=True)
test_df = test.groupby("label").head(50).reset_index(drop=True) if SMOKE else test.reset_index(drop=True)
pool_final = pool_df.to_dict("records")
test_final = test_df.to_dict("records")
test_pairs = build_pairs(test_final)

if os.getenv("RUN_FINAL", "1") == "1":
    for run in FINAL:
        if not FORCE and run.run_id in done_ids(LEDGER):
            print("lewati, sudah ada:", run.run_id)
            continue
        set_seed(run.seed)
        tr_loader = make_loader(pool_final, run, shuffle=True)
        te_loader = make_loader(test_final, run, shuffle=False)
        model = build_model(run).to(DEVICE)
        crit = nn.CrossEntropyLoss(weight=class_weights(pool_final))
        opt = torch.optim.AdamW(model.parameters(), lr=run.lr, weight_decay=0.01)
        scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
        steps = len(tr_loader) * epochs_final
        sched = get_cosine_schedule_with_warmup(
            opt, num_warmup_steps=int(WARMUP * steps), num_training_steps=steps
        )
        torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()

        for epoch in range(1, epochs_final + 1):
            model.train()
            for batch in tr_loader:
                batch = move(batch)
                opt.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
                    loss = crit(model(batch), batch["label"])
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                sched.step()
            print(f"  {run.run_id} epoch {epoch}/{epochs_final}")

        wall = time.perf_counter() - t0
        vram = torch.cuda.max_memory_allocated() / 1e6 if DEVICE.type == "cuda" else 0.0
        y, p, ids = predict(model, te_loader)
        torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, RUNS_DIR / f"{run.run_id}.pt")
        np.savez(RUNS_DIR / f"{run.run_id}-test.npz", ids=ids, y=y, prob=p)
        sc = score(y, p.argmax(1))
        append_ledger(
            run.row()
            | {
                "epochs_run": epochs_final,
                "epoch_best": epochs_final,
                "acc": round(sc["acc"], 4),
                "bal_acc": round(sc["bal_acc"], 4),
                "f1_macro": round(sc["f1_macro"], 4),
                "f1_true": round(sc["f1_true"], 4),
                "f1_ooc": round(sc["f1_ooc"], 4),
                "f1_mc": round(sc["f1_mc"], 4),
                "val_n": len(test_final),
                "wall_clock_s": round(wall, 1),
                "params": count_params(model),
                "vram_peak_mb": round(vram, 0),
                "catatan": "test-700",
            },
            LEDGER,
        )
        del model, opt, crit, scaler, tr_loader, te_loader, sched
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
else:
    print("RUN_FINAL mati")

led_z = pd.DataFrame(read_ledger(LEDGER))
for col in ["acc", "bal_acc", "f1_macro", "f1_true", "f1_ooc", "f1_mc"]:
    led_z[col] = pd.to_numeric(led_z[col], errors="coerce")
z_rows = led_z[led_z.kode.isin(["Z1", "Z2", "Z3"])].drop_duplicates("kode").sort_values("kode").copy()
if len(z_rows):
    pw_z = []
    for r_id in z_rows["run_id"]:
        npz_path = RUNS_DIR / f"{r_id}-test.npz"
        if test_pairs and npz_path.exists():
            prob_z = np.load(npz_path)["prob"][:, 0]
            pw_z.append(float(np.mean([prob_z[a] > prob_z[b] for a, b in test_pairs])))
        else:
            pw_z.append(float("nan"))
    z_rows["pairwise"] = pw_z
    display(z_rows[["kode", "seed", "acc", "bal_acc", "f1_macro", "f1_true", "f1_ooc", "f1_mc", "pairwise"]])
    stats_z = pd.DataFrame({
        "mean": z_rows[["acc", "bal_acc", "f1_macro", "f1_true", "f1_ooc", "f1_mc", "pairwise"]].mean().round(4),
        "std": z_rows[["acc", "bal_acc", "f1_macro", "f1_true", "f1_ooc", "f1_mc", "pairwise"]].std(ddof=1).round(4),
    })
    display(stats_z)

final config: indobert clip xattn_uni | lr = 1e-05 | do = 0.1 | epochs = 7


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Z1_indobert_clip_xattn_uni_s727 epoch 1/7
  Z1_indobert_clip_xattn_uni_s727 epoch 2/7
  Z1_indobert_clip_xattn_uni_s727 epoch 3/7
  Z1_indobert_clip_xattn_uni_s727 epoch 4/7
  Z1_indobert_clip_xattn_uni_s727 epoch 5/7
  Z1_indobert_clip_xattn_uni_s727 epoch 6/7
  Z1_indobert_clip_xattn_uni_s727 epoch 7/7


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

  Z3_indobert_clip_xattn_uni_s729 epoch 1/7
  Z3_indobert_clip_xattn_uni_s729 epoch 2/7
  Z3_indobert_clip_xattn_uni_s729 epoch 3/7
  Z3_indobert_clip_xattn_uni_s729 epoch 4/7
  Z3_indobert_clip_xattn_uni_s729 epoch 5/7
  Z3_indobert_clip_xattn_uni_s729 epoch 6/7
  Z3_indobert_clip_xattn_uni_s729 epoch 7/7


,kode,seed,acc,bal_acc,f1_macro,f1_true,f1_ooc,f1_mc,pairwise
26,Z1,727,0.7100,0.7209,0.7166,0.6911,0.7070,0.7517,0.840278
27,Z2,728,0.7129,0.7233,0.7206,0.7131,0.6789,0.7698,0.833333
28,Z3,729,0.7171,0.7280,0.7240,0.7123,0.6962,0.7635,0.819444


,mean,std
acc,0.7133,0.0036
bal_acc,0.7241,0.0036
f1_macro,0.7204,0.0037
f1_true,0.7055,0.0125
f1_ooc,0.6940,0.0142
f1_mc,0.7617,0.0092
pairwise,0.8310,0.0106


## 12. Analisis error dan ringkasan

**Tujuan** menjelaskan jenis kesalahan dan batasannya.

**Input** prediksi test tiga seed dan ledger.

**Output** confusion matrix, contoh kesalahan, dan ringkasan.

**Keputusan** kelas MC dilaporkan bersama batasan hasil audit, karena sebagian item MC
tidak bisa diperiksa dari gambar.

In [14]:
led = pd.DataFrame(read_ledger(LEDGER))
for col in ["acc", "bal_acc", "f1_macro", "f1_true", "f1_ooc", "f1_mc", "params", "wall_clock_s"]:
    led[col] = pd.to_numeric(led[col], errors="coerce")

ringkas = led.drop_duplicates("run_id")[["kode", "faktor", "teks", "gambar", "fusion", "lr", "dropout", "seed", "acc", "bal_acc", "f1_macro", "f1_mc", "params"]]
display(ringkas.sort_values(["faktor", "kode", "seed"]))

test_df = test.groupby("label").head(50).reset_index(drop=True) if SMOKE else test.reset_index(drop=True)
test_pairs = build_pairs(test_df.to_dict("records"))
test_files = sorted(RUNS_DIR.glob("Z*-test.npz"))

if test_files:
    probs = np.mean([np.load(f)["prob"] for f in test_files], axis=0)
    y = np.load(test_files[0])["y"]
    assert len(y) == len(test_df), f"jumlah test tidak cocok: {len(y)} vs {len(test_df)}"
    pred = probs.argmax(1)
    sc = score(y, pred)
    ens_pw = float(np.mean([probs[a, 0] > probs[b, 0] for a, b in test_pairs])) if test_pairs else float("nan")
    print("ensemble 3 seed test:", {k: round(v, 4) for k, v in sc.items() if k != "confusion"} | {"pairwise": round(ens_pw, 4)})

    cm = np.array(sc["confusion"])
    display(pd.DataFrame(cm, index=[f"aktual:{n}" for n in LABEL_NAMES], columns=[f"pred:{n}" for n in LABEL_NAMES]))

    fig, ax = plt.subplots(figsize=(5.4, 4.4))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(3), LABEL_NAMES, rotation=20)
    ax.set_yticks(range(3), LABEL_NAMES)
    ax.set_xlabel("Prediksi")
    ax.set_ylabel("Aktual")
    ax.set_title("Confusion Matrix Ensemble Uji 700")
    for r_i in range(3):
        for c_i in range(3):
            ax.text(c_i, r_i, str(cm[r_i, c_i]), ha="center", va="center", color="white" if cm[r_i, c_i] > cm.max() / 2 else "black")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(FIG / ("12_confusion_smoke.png" if SMOKE else "12_confusion.png"), dpi=160)
    plt.close(fig)

    err = test_df.assign(pred=[LABEL_NAMES[i] for i in pred], conf=probs.max(1))
    err = err[err.label != err.pred].sort_values("conf", ascending=False)
    print(f"total salah prediksi: {len(err)} dari {len(test_df)} ({len(err)/len(test_df):.1%})")
    display(err[["label", "pred", "conf", "category", "caption"]].head(12))
else:
    print("belum ada prediksi test Z*-test.npz")

audit_path = EXPERIMENTS / "audit" / "hasil_audit.json"
if audit_path.exists():
    audit = json.loads(audit_path.read_text())
    print(
        f"catatan audit manusia: presisi tertimbang TRUE = {audit['presisi_tertimbang']:.4f} "
        f"(CI95% gabungan {audit['ci_gabungan'][0]:.4f}-{audit['ci_gabungan'][1]:.4f}; "
        f"{len(audit['tidak_cocok'])} tidak cocok, {len(audit['tidak_jelas'])} ragu dari {audit['item']} item). "
        "Sebagian item miscaptioned mengubah entitas non-visual sehingga f1_mc perlu dibaca bersama batasan ini."
    )

,kode,faktor,teks,gambar,fusion,lr,dropout,seed,acc,bal_acc,f1_macro,f1_mc,params
11,M1,encoder,indobert,clip,xattn_uni,2e-05,0.1,727,0.6268,0.6408,0.6346,0.6810,172631299.0
12,M2,encoder,indobert,dinov2,xattn_uni,2e-05,0.1,727,0.5991,0.6163,0.6136,0.6789,172556803.0
13,M3,encoder,mbert,clip,xattn_uni,2e-05,0.1,727,0.5459,0.5770,0.5606,0.6280,172631299.0
14,M4,encoder,mbert,dinov2,xattn_uni,2e-05,0.1,727,0.5528,0.5844,0.5690,0.6443,172556803.0
26,Z1,final,indobert,clip,xattn_uni,1e-05,0.1,727,0.7100,0.7209,0.7166,0.7517,172631299.0
27,Z2,final,indobert,clip,xattn_uni,1e-05,0.1,728,0.7129,0.7233,0.7206,0.7698,172631299.0
28,Z3,final,indobert,clip,xattn_uni,1e-05,0.1,729,0.7171,0.7280,0.7240,0.7635,172631299.0
6,F0,fusion,mbert,dinov2,late,2e-05,0.1,727,0.4861,0.5357,0.4999,0.6167,172161542.0
1,F1,fusion,mbert,dinov2,concat,2e-05,0.1,727,0.5289,0.5616,0.5466,0.6275,172293123.0
7,F2,fusion,mbert,dinov2,gated,2e-05,0.1,727,0.5671,0.5865,0.5754,0.6152,172358403.0


ensemble 3 seed test: {'acc': 0.7314, 'bal_acc': 0.7411, 'f1_macro': 0.7376, 'f1_true': 0.7251, 'f1_ooc': 0.7148, 'f1_mc': 0.7729, 'pairwise': 0.8542}


,pred:true,pred:out-of-context,pred:miscaptioned
aktual:true,215,42,21
aktual:out-of-context,79,183,16
aktual:miscaptioned,21,9,114


total salah prediksi: 188 dari 700 (26.9%)


,label,pred,conf,category,caption
468,out-of-context,miscaptioned,0.993386,regional,Peristiwa longsor yang terjadi di terowongan p...
143,true,miscaptioned,0.992290,otomotif,Kecelakaan beruntun di terowongan Pasar Rebo T...
216,true,miscaptioned,0.992078,regional,Gubuk di perkebunan sawit di Kabupaten Lampung...
186,true,miscaptioned,0.991868,regional,Tim BPBD melakukan assement bangunan madrasah ...
464,out-of-context,miscaptioned,0.991825,regional,Tim BPBD melakukan assement bangunan madrasah ...
494,out-of-context,miscaptioned,0.990832,regional,Gubuk di perkebunan sawit di Kabupaten Lampung...
164,true,miscaptioned,0.990798,regional,Ilustrasi kecelakaan bus. Kondisi korban WNI k...
421,out-of-context,miscaptioned,0.990712,otomotif,Kecelakaan beruntun di terowongan Pasar Rebo T...
182,true,miscaptioned,0.989204,megapolitan,"Tak hanya lari dan jalan santai, pengunjung Ge..."
517,out-of-context,miscaptioned,0.988384,bandung,Sebuah papan imbauan agar truk besar dengan su...


catatan audit manusia: presisi tertimbang TRUE = 0.9331 (CI95% gabungan 0.8891-0.9678; 11 tidak cocok, 7 ragu dari 180 item). Sebagian item miscaptioned mengubah entitas non-visual sehingga f1_mc perlu dibaca bersama batasan ini.
